In [1]:
import re
import pandas as pd

In [ ]:
df = pd.read_csv('Rebuttal_nat_rea/GRPO_results_ben.csv')
# df = pd.read_csv('base_Qwen2_5_imgonly.csv')

print(df.columns)

In [46]:
predictions = df['predictions'].tolist()
ground_truths = df['solution'].tolist()
# predictions = df['prediction_answer'].tolist()
# ground_truths = df['ground_truth_answer'].tolist()

In [47]:
answer_pattern = re.compile(
    r"""
    <answer>\s*(?P<answer>.*?)\s*</answer>              # Capture content between answer tags
    .*?                                                 # Match any noise/whitespace between tags
    (?:<reasoning>\s*(?P<reasoning>.*?)\s*</reasoning>)? # Capture content between reasoning tags (Optional)
    """,
    re.DOTALL | re.VERBOSE | re.IGNORECASE)

predictions_answer = []
predictions_reasoning = []

for pred in predictions:
    match = answer_pattern.search(pred)
    if match:
        predictions_answer.append(match.group("answer").strip() if match.group("answer") else "")
        predictions_reasoning.append(match.group("reasoning").strip() if match.group("reasoning") else "")
    else:
        predictions_answer.append("")
        predictions_reasoning.append("")


ground_truth_answer = []
ground_truth_reasoning = []

for gt in ground_truths:
    match = answer_pattern.search(gt)
    if match:
        ground_truth_answer.append(match.group("answer").strip() if match.group("answer") else "")
        ground_truth_reasoning.append(match.group("reasoning").strip() if match.group("reasoning") else "")
    else:
        ground_truth_answer.append("")
        ground_truth_reasoning.append("")

In [49]:
# answer_option_pattern = re.compile(r"([ABCD])(\.)?\s*", re.IGNORECASE)
# temp_gt = [gt[:2] for gt in ground_truth_answer ]
# temp_pred = [pred[:2] for pred in predictions_answer]
# gt_options = [1 if answer_option_pattern.search(gt) else 0 for gt in temp_gt]
# pred_options = [1 if answer_option_pattern.search(pred) else 0 for pred in temp_pred]
# print(sum(gt_options) / len(gt_options))
# print(sum(pred_options) / len(pred_options))
# # for idx, p in enumerate(pred_options):
# #     if p != 1:
# #         print(idx, p)

# gt_options2 = [answer_option_pattern.search(gt).group(0)[:1] if answer_option_pattern.search(gt) else "" for gt in temp_gt]
# pred_options2 = [answer_option_pattern.search(pred).group(0)[:1] if answer_option_pattern.search(pred) else "" for pred in temp_pred]
# correct = [1 if g == p else 0 for g, p in zip(gt_options2, pred_options2)]
# print(gt_options2)
# print(pred_options2)
# print(sum(correct)/len(correct))

In [ ]:
import re

option_label_map = {
    "english": {"a": "a", "b": "b", "c": "c", "d": "d"},
    "hindi":   {"a": "क", "b": "ख", "c": "ग", "d": "घ"},
    "bengali": {"a": "ক", "b": "খ", "c": "গ", "d": "ঘ"},
    "marathi": {"a": "क", "b": "ख", "c": "ग", "d": "घ"},
    "gujarati":{"a": "ક", "b": "ખ", "c": "ગ", "d": "ઘ"},
    "tamil":   {"a": "௧", "b": "௨", "c": "௩", "d": "௪"},
}

# Build a single regex that matches any option label across languages,
# allowing optional "." and whitespace after it (and tolerant to "a/A" etc).
all_labels = sorted(
    {lbl for lang in option_label_map.values() for lbl in lang.values()},
    key=len,
    reverse=True,  # longer first (safer for regex alternation)
)
label_pattern = re.compile(rf"^({'|'.join(map(re.escape, all_labels))})(\.)?\s*", re.UNICODE)

# Reverse map: label -> canonical option key (a/b/c/d)
label_to_key = {}
for lang, mapping in option_label_map.items():
    for k, lbl in mapping.items():
        # If a label appears in multiple langs (e.g., क/ख/ग/घ), it's the same meaning anyway.
        # We keep first occurrence; it maps to the same canonical key.
        label_to_key.setdefault(lbl, k)

def extract_option_key(text: str) -> str:
    """Return canonical option key: 'a'/'b'/'c'/'d' or '' if not found."""
    if not text:
        return ""
    s = str(text).strip()
    m = label_pattern.search(s)
    if not m:
        return ""
    label = m.group(1)
    # Handle English letters robustly (A/a etc)
    if label.lower() in {"a", "b", "c", "d"}:
        return label.lower()
    return label_to_key.get(label, "")

# ----- Your evaluation (multilingual) -----
# (Keeping your original slicing idea; you can also remove [:2] if answers are longer)
temp_gt   = [str(gt)[:2] for gt in ground_truth_answer]
temp_pred = [str(pred)[:2] for pred in predictions_answer]

gt_keys   = [extract_option_key(gt) for gt in temp_gt]
pred_keys = [extract_option_key(pred) for pred in temp_pred]

gt_has   = [1 if k else 0 for k in gt_keys]
pred_has = [1 if k else 0 for k in pred_keys]

print(sum(gt_has) / len(gt_has))
print(sum(pred_has) / len(pred_has))

correct = [1 if g == p and g != "" else 0 for g, p in zip(gt_keys, pred_keys)]
print(gt_keys)
print(pred_keys)
print("Accuracy :" , sum(correct) / len(correct))


In [ ]:
print(correct)